# Transfer Learning for Pneumonia Detection
## Chest X-ray Classification with PyTorch

This notebook provides a **mini walkthrough** of the full pipeline:
1. Download dataset via KaggleHub
2. Explore & visualise the data
3. Build a Transfer Learning model (ResNet-50)
4. Train on a small subset
5. Evaluate and visualise results

## 1. Imports & Device Setup

In [ ]:
import os, random, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from tqdm.notebook import tqdm
import kagglehub

# ── Reproducibility ──────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Device ───────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Download Dataset

In [ ]:
# Run data_download.py to fetch and place the dataset under data/chest_xray/
# (safe to re-run — skips download if already present)
import sys, subprocess
subprocess.run([sys.executable, 'data_download.py'], check=True)

from data_download import DEST_PATH
data_root = DEST_PATH
print('Dataset root:', data_root)
print('Contents:', sorted([p.name for p in data_root.iterdir()]))

## 3. Dataset Exploration

In [ ]:
CLASSES = ["NORMAL", "PNEUMONIA"]

# Count images per class per split
for split in ["train", "val", "test"]:
    split_path = data_root / split
    counts = {cls: len(list((split_path / cls).glob("*.jpeg")) +
                        list((split_path / cls).glob("*.jpg")) +
                        list((split_path / cls).glob("*.png")))
              for cls in CLASSES}
    print(f"{split:5s}: {counts}")

# ── Class distribution bar chart ─────────────
train_counts = {cls: len(list((data_root / "train" / cls).glob("*"))) for cls in CLASSES}
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(CLASSES, train_counts.values(), color=["steelblue", "tomato"], edgecolor="black")
ax.bar_label(bars, padding=3)
ax.set_title("Training Set Class Distribution")
ax.set_ylabel("Number of Images")
plt.tight_layout(); plt.show()

In [ ]:
# ── Sample images from each class ───────────
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, cls in enumerate(CLASSES):
    cls_path = data_root / "train" / cls
    imgs = list(cls_path.glob("*.jpeg"))[:4]
    for col, img_path in enumerate(imgs):
        img = Image.open(img_path).convert("RGB")
        axes[row][col].imshow(img, cmap="gray")
        axes[row][col].set_title(cls, fontsize=10)
        axes[row][col].axis("off")
plt.suptitle("Sample Chest X-ray Images", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

## 4. Data Transforms & Loaders

In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 16        # small for notebook demo
NUM_WORKERS = 2
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_train = datasets.ImageFolder(data_root / "train", train_tf)
full_val   = datasets.ImageFolder(data_root / "val",   val_tf)
full_test  = datasets.ImageFolder(data_root / "test",  val_tf)

# ── Use a subset for the quick notebook demo ─
SUBSET_SIZE = 300   # change to None for full training
if SUBSET_SIZE:
    idx = random.sample(range(len(full_train)), SUBSET_SIZE)
    train_ds = Subset(full_train, idx)
else:
    train_ds = full_train

loaders = {
    "train": DataLoader(train_ds,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True),
    "val":   DataLoader(full_val,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True),
    "test":  DataLoader(full_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True),
}

print(f"Train batches : {len(loaders['train'])}")
print(f"Val batches   : {len(loaders['val'])}")
print(f"Test batches  : {len(loaders['test'])}")
print(f"Classes       : {full_train.classes}")

## 5. Visualise Augmented Training Batch

In [ ]:
def denormalise(tensor):
    """Undo ImageNet normalisation for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    return (tensor * std + mean).clamp(0, 1)

images, labels = next(iter(loaders["train"]))
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i, ax in enumerate(axes.flat):
    if i >= len(images): ax.axis("off"); continue
    img = denormalise(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(full_train.classes[labels[i].item()], fontsize=8)
    ax.axis("off")
plt.suptitle("Augmented Training Batch", fontsize=12)
plt.tight_layout(); plt.show()

## 6. Build the Transfer Learning Model (ResNet-50)

In [ ]:
NUM_CLASSES  = 2
FREEZE_BACKBONE = True   # set False to fine-tune all layers

# Load pretrained ResNet-50
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

if FREEZE_BACKBONE:
    for param in model.parameters():
        param.requires_grad = False

# Replace classification head
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(in_features, NUM_CLASSES),
)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")
print(model.fc)

## 7. Training (Mini Demo – 5 Epochs)

In [ ]:
import copy

NUM_EPOCHS        = 5       # quick demo; use 20+ for full training
LR                = 1e-4
WEIGHT_DECAY      = 1e-4
ES_PATIENCE       = 3       # early stopping patience (epochs)
ES_MIN_DELTA      = 1e-4    # minimum val-acc improvement to reset patience

# Weighted loss for class imbalance
labels_list = [s[1] for s in full_train.samples]
counts      = np.bincount(labels_list)
weights     = torch.tensor(1.0 / counts, dtype=torch.float32).to(device)
criterion   = nn.CrossEntropyLoss(weight=weights)

optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# ── Training loop (with early stopping) ──────────────────────────────────
history           = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_acc          = 0.0
best_weights      = copy.deepcopy(model.state_dict())
epochs_no_improve = 0

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    early_stop = False

    for phase in ("train", "val"):
        model.train() if phase == "train" else model.eval()
        running_loss, running_corrects = 0.0, 0

        for inputs, labels in tqdm(loaders[phase], desc=f"  {phase}", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.set_grad_enabled(phase == "train"):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                if phase == "train":
                    loss.backward(); optimizer.step()

            n = inputs.size(0)
            running_loss     += loss.item() * n
            running_corrects += (preds == labels).sum().item()

        if phase == "train":
            scheduler.step()

        size       = len(loaders[phase].dataset)
        epoch_loss = running_loss / size
        epoch_acc  = running_corrects / size
        history[f"{phase}_loss"].append(epoch_loss)
        history[f"{phase}_acc"].append(epoch_acc)
        print(f"  {phase:5s}  loss={epoch_loss:.4f}  acc={epoch_acc:.4f}")

        if phase == "val":
            if epoch_acc >= best_acc + ES_MIN_DELTA:
                best_acc          = epoch_acc
                best_weights      = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
                print(f"  ✓ New best val acc: {best_acc:.4f}")
            else:
                epochs_no_improve += 1
                print(f"  No improvement for {epochs_no_improve}/{ES_PATIENCE} epochs")
                if epochs_no_improve >= ES_PATIENCE:
                    print(f"  Early stopping triggered.")
                    early_stop = True

    if early_stop:
        break

print(f"\nBest val accuracy: {best_acc:.4f}")
model.load_state_dict(best_weights)

## 8. Training Curves

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history["train_loss"], "o-", label="Train")
axes[0].plot(epochs, history["val_loss"],   "s-", label="Val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(epochs, history["train_acc"], "o-", label="Train")
axes[1].plot(epochs, history["val_acc"],   "s-", label="Val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()

plt.tight_layout(); plt.show()

## 9. Fine-tuning (Optional)

Set `ENABLE_FINETUNE = True` to unfreeze the full ResNet-50 backbone and
continue training with a smaller learning rate. This can squeeze out extra
accuracy after the classification head has converged.

> Skip this cell if you want to go straight to evaluation.

In [ ]:
ENABLE_FINETUNE  = False    # ← set True to run
FINETUNE_EPOCHS  = 5
FINETUNE_LR      = 1e-5
FT_ES_PATIENCE   = 3

if ENABLE_FINETUNE:
    print("Unfreezing all layers for fine-tuning…")
    for p in model.parameters():
        p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,}")

    ft_optimizer = optim.AdamW(model.parameters(), lr=FINETUNE_LR, weight_decay=WEIGHT_DECAY)
    ft_scheduler = optim.lr_scheduler.StepLR(ft_optimizer, step_size=3, gamma=0.1)

    ft_history        = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    ft_best_acc       = best_acc
    ft_best_weights   = copy.deepcopy(model.state_dict())
    ft_no_improve     = 0

    for epoch in range(FINETUNE_EPOCHS):
        print(f"\n[FT] Epoch {epoch+1}/{FINETUNE_EPOCHS}")
        ft_early_stop = False

        for phase in ("train", "val"):
            model.train() if phase == "train" else model.eval()
            running_loss, running_corrects = 0.0, 0

            for inputs, labels in tqdm(loaders[phase], desc=f"  {phase}", leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                ft_optimizer.zero_grad()
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == "train":
                        loss.backward(); ft_optimizer.step()

                n = inputs.size(0)
                running_loss     += loss.item() * n
                running_corrects += (preds == labels).sum().item()

            if phase == "train":
                ft_scheduler.step()

            size       = len(loaders[phase].dataset)
            epoch_loss = running_loss / size
            epoch_acc  = running_corrects / size
            ft_history[f"{phase}_loss"].append(epoch_loss)
            ft_history[f"{phase}_acc"].append(epoch_acc)
            print(f"  {phase:5s}  loss={epoch_loss:.4f}  acc={epoch_acc:.4f}")

            if phase == "val":
                if epoch_acc >= ft_best_acc + ES_MIN_DELTA:
                    ft_best_acc     = epoch_acc
                    ft_best_weights = copy.deepcopy(model.state_dict())
                    ft_no_improve   = 0
                    print(f"  ✓ New best val acc: {ft_best_acc:.4f}")
                else:
                    ft_no_improve += 1
                    print(f"  No improvement for {ft_no_improve}/{FT_ES_PATIENCE} epochs")
                    if ft_no_improve >= FT_ES_PATIENCE:
                        print("  Fine-tuning early stopping triggered.")
                        ft_early_stop = True

        if ft_early_stop:
            break

    model.load_state_dict(ft_best_weights)
    print(f"\nFine-tuning complete. Best val accuracy: {ft_best_acc:.4f}")

    # Merge histories for a unified plot
    for key in history:
        history[key].extend(ft_history[key])
    best_acc = ft_best_acc
else:
    print("Fine-tuning skipped (ENABLE_FINETUNE=False).")

## 10. Test Set Evaluation

In [ ]:
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for inputs, labels in tqdm(loaders["test"], desc="Testing"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        probs   = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

print("── Classification Report ──────────────────────")
print(classification_report(all_labels, all_preds, target_names=CLASSES))
print(f"ROC-AUC: {roc_auc_score(all_labels, all_probs):.4f}")

## 11. Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_ylabel("Actual"); axes[0].set_xlabel("Predicted")
axes[0].set_title("Confusion Matrix")

# ROC curve
auc = roc_auc_score(all_labels, all_probs)
fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, lw=2, label=f"AUC = {auc:.4f}")
axes[1].plot([0,1],[0,1],"k--")
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
axes[1].set_title("ROC Curve"); axes[1].legend()

plt.tight_layout(); plt.show()

## 12. Single-Image Inference

In [ ]:
def predict_single(img_path: str):
    """Run inference on a single chest X-ray image."""
    img = Image.open(img_path).convert("RGB")
    tensor = val_tf(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()
    pred_class = CLASSES[int(np.argmax(probs))]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img, cmap="gray")
    axes[0].set_title(f"Input Image"); axes[0].axis("off")

    colour = ["steelblue", "tomato"]
    bars   = axes[1].barh(CLASSES, probs, color=colour)
    axes[1].bar_label(bars, fmt="%.3f", padding=3)
    axes[1].set_xlim(0, 1.1)
    axes[1].set_title(f"Prediction: {pred_class}  ({probs.max()*100:.1f}%)")
    plt.tight_layout(); plt.show()
    return pred_class, probs

# Demo: pick a random test image
test_dir = data_root / "test"
sample_cls = random.choice(CLASSES)
sample_imgs = list((test_dir / sample_cls).glob("*.jpeg"))
if not sample_imgs:
    sample_imgs = list((test_dir / sample_cls).glob("*.jpg"))
sample_img = random.choice(sample_imgs)
print(f"Ground truth: {sample_cls}")
predict_single(str(sample_img))

## 13. Save Model Checkpoint

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)
torch.save(model.state_dict(), "outputs/best_model.pth")
print("Checkpoint saved → outputs/best_model.pth")
print("\nTo run the full pipeline with all epochs:")
print("  python main.py --model resnet50 --epochs 20")
print("\nTo launch the Gradio demo:")
print("  python app.py")